# Inference smoke test

Loads the FastAPI service against locally-present `models/<pair>/` artifacts and
exercises `/health`, `/translate`, and `/translate/batch`.

In [ ]:
# Cell 1 - locate project root
import os
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

project_root = Path(os.getenv("PROJECT_ROOT", "").strip()) if os.getenv("PROJECT_ROOT") else None
candidates = [project_root, Path.cwd(), Path("/content/drive/MyDrive/Transformer")]
ROOT = next((c for c in candidates if c is not None and (c / "infer.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root with infer.py. Set PROJECT_ROOT.")
os.chdir(ROOT)
print("CWD:", Path.cwd())

In [ ]:
# Cell 2 - (optional) unpack a downloaded Colab artifact zip into models/
from pathlib import Path
import zipfile

zip_candidates = [Path.cwd() / "models_artifact.zip", Path.cwd() / "models" / "models_artifact.zip"]
zip_file = next((p for p in zip_candidates if p.exists()), None)
if zip_file is None:
    print("No models_artifact.zip found (fine if models/<pair>/ already exists).")
else:
    target = Path.cwd() / "models"
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_file) as zf:
        zf.extractall(target)
    print("Extracted into", target)

In [ ]:
# Cell 3 - load the service
import importlib, sys
from pathlib import Path

os.environ.setdefault("MODELS_DIR", str((Path.cwd() / "models").resolve()))
os.environ.setdefault("DEFAULT_NUM_BEAMS", "1")
for name in ("infer", "serve_model"):
    sys.modules.pop(name, None)
infer = importlib.import_module("infer")
print("MODELS_DIR:", infer.MODELS_DIR, "| DEVICE:", infer.DEVICE)

In [ ]:
# Cell 4 - exercise the endpoints
from fastapi.testclient import TestClient

with TestClient(infer.app) as client:
    health = client.get("/health").json()
    print("Health:", health)

    if health.get("status") != "ok":
        raise RuntimeError("No artifacts loaded. Train (train.py) or unpack the Colab zip first.")

    for lang, text in [("ca", "El perill era desesperat."), ("de", "Das ist ein Test.")]:
        if lang in health["supported_langs"]:
            r = client.post("/translate", json={"text": text, "src_lang": lang, "num_beams": 3})
            print(lang, "->", r.status_code, r.json())

    batch = client.post("/translate/batch", json={"items": [
        {"text": "El perill era desesperat.", "src_lang": "ca", "num_beams": 1},
        {"text": "Das ist ein Test.", "src_lang": "de", "num_beams": 1},
    ]})
    print("Batch:", batch.status_code, batch.json())